# Nik Studio - animate

**You give it pictures and a song. It gives you one finished MP4.**

### Check first. It is free.

Colab charges for a GPU from the moment it connects, whatever you run on
it. So do the checking with no GPU at all:

1. **Runtime > Change runtime type > CPU**
2. Run **cell 2**. It finds your files, checks every one of them, and
   stops with `this cost nothing`.
3. Only once it says that: **Runtime > Change runtime type > L4 GPU**,
   run **cell 1**, then **cell 2** again.

A CPU runtime costs nothing, so a wrong folder or a picture that will
not open is found for free instead of on a meter.

Pick **L4**, not A100. The model is small and A100 spends units far
faster for no better result.

### Before you start

Make this folder in Google Drive and put your files in it:

```
My Drive / NikStudio / Input /
      Scene01.png
      Scene02.png
      Scene03.png
      song.mp3
```

Any names work - the pictures are used **in order**, so number them, and
any one audio file is taken as the song. `python tools\prepare.py --copy`
builds this folder for you and checks it before you ever open Colab.

The finished video comes back as:

```
My Drive / NikStudio / Output / Episode.mp4
```

### What it does

Every picture becomes a moving clip, the clips are cut to share the
length of the song exactly, and the song is laid over the top.

It reads the card and picks its own quality settings, so there is
nothing to tune. On a card with room for it that means **Wan 2.2**,
which is trained on movement - the earlier model was trained to hold a
picture still, which is exactly what was wrong with the videos it made.
Sixteen shots take about an hour.

It finishes by printing a **Movement** number for the video and for one
clip straight from the model. Send those two numbers back: they say
where a problem is, and they fit in a message where a video does not.

It saves each clip to Drive as it finishes. If the session dies, run it
again - it picks up where it stopped instead of starting over. A picture
you replace is noticed and made again; the rest are not.

**Honest about one thing:** the mouth moves, but it is not lip-synced to
the words. Nothing free does real lip-sync yet.


In [ ]:
# ======================================================================
# CELL 1 of 2 - the packages.  Only needed once a GPU is turned on
# ======================================================================
#
# Skip this while you are still checking on a CPU runtime - cell 2 does
# the checking with what Colab already has.
#
# If Colab offers "RESTART SESSION" when this finishes, click it.
# Cell 2 depends on nothing in here, so a restart costs nothing.

!pip install -q "diffusers>=0.35" "transformers>=4.44" accelerate safetensors sentencepiece bitsandbytes imageio-ffmpeg librosa ftfy

print("Packages installed. Now run cell 2.")


In [ ]:
# ======================================================================
# CELL 2 of 2 - the whole thing
# ======================================================================
#
# Finds your pictures and your song in Drive, animates every picture,
# cuts the clips to share the song's length, lays the song over the top,
# and plays you the result.
#
# It depends on nothing above it, so running cells out of order or
# letting Colab restart the runtime cannot break it.

import gc
import json
import os
import re
import shutil
import subprocess
import sys
import time

from pathlib import Path

# Let the allocator grow its blocks instead of demanding one big
# contiguous run. On a card this full, fragmentation alone can be the
# difference between fitting and not.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

from PIL import Image


# ======================================================================
# SETTINGS - the only part worth editing
# ======================================================================

# Which notebook this is.
#
# Printed first, before anything else, because there is no other way to
# tell one from another once it is open in Colab - and a clip made by
# an older one looks exactly like a clip the new one refused to change.
# An hour went into working out that a video sent back for review was
# byte for byte the one sent the day before.
#
# If you are reading a run and this line is not the one you expect, the
# notebook in Colab is not the notebook you were given.
BUILD = "2026-09-03 - Split of, so two of them can share a frame"

DRIVE = "/content/drive/MyDrive"

FOLDER = DRIVE + "/NikStudio"

# Where the work goes when Drive will not connect. This session's own
# disk, which is emptied when the session ends.
LOCAL = "/content/NikStudio"

# What HAPPENS in the shot. Not what the picture shows - that is already
# there. One clear physical action beats three vague ones.
# One action for the character - and then a plain statement that
# everything else stays put.
#
# Whatever is not named drifts. Naming only the boy left the puppy, the
# kitten and the duckling free to melt, and they did, from about a
# second and a half in. And dropping "the camera does not move" let the
# whole frame push slowly in on its own.
#
# So: one movement, then who else is in the shot, then the camera.
PROMPT = (
    "The little boy sways gently from side to side, smiling. "
    "The puppy, the kitten and the duckling stay where they are, "
    "watching him. The background does not change. "
    "The camera does not move."
)

# A picture can have its own, by file name. Anything not listed here
# uses PROMPT above.
PROMPTS = {
    # "Scene02.png": "The boy splashes the water with both hands, ...",
}

# ------------------------------------------------- writing the scenes

# Put a script.txt in the Input folder - one scene a line - and the
# shots are made from your words instead of from pictures:
#
#     Nik waves at the puppy while the flowers sway around them
#     Nik runs down the garden path, the kitten chasing him
#
# Nothing is generated from a picture then, so nothing anchors the look.
# These two lines are what hold it together instead: the character is
# described the same way in every shot, and so is the style. Keep them
# exact and unchanging - a word altered between runs is a different boy.
CHARACTER = (
    "Nik, a chubby cheerful 3 year old Indian toddler boy, warm brown "
    "skin, soft wavy dark brown hair, very large expressive brown eyes "
    "with long lashes, round rosy cheeks, small button nose, wearing a "
    "bright blue t-shirt under yellow denim dungarees and blue canvas "
    "sneakers"
)

# The same three things, said short, for the drawing model.
#
# SDXL reads 77 tokens and throws the rest away without stopping. The
# video prompt is about 225, so the first attempt at drawing got the
# action and nothing else: no framing, no character, no style, all of
# it silently cut. The clue was one warning line in a wall of them.
#
# The character is not repeated here on purpose - that is what the
# reference picture is for, and spending forty of seventy-seven tokens
# describing a boy whose photograph is already in front of the model is
# what pushed the style off the end.
# "full body, centred" was not enough - the drawing came back with his
# legs cut off at the shins, one hand off the right edge and his head
# against the top. Say what "full body" means, in words a picture can
# be checked against.
# ======================================================================
# THE SHOW - the six settings that change when the video changes
# ======================================================================
#
# Everything from here to DRAW_STYLE is about YOUR show and nothing
# else. A different character, a different place, a different song:
# change these six and the rest of the notebook does not care.
#
#   WHO           who the character is, short - for the drawings
#   CHARACTER     the same person said at length - for the video
#   PLACE         where the whole video happens
#   SCENERY       words that mean weather rather than somebody
#   DRAW_FRAMING  how far away the camera stands
#   DRAW_STYLE    what it looks like
#
# WHO and CHARACTER are the same person twice, at two lengths, because
# the drawing model reads 77 tokens and the video model reads 512. Keep
# them agreeing - a colour in one and not the other is two children.


# Who the boy is, in words. This is what holds him together now.
#
# Every one of these words attaches to "boy" and cannot wander onto a
# kitten, which is the whole reason it is here rather than in a
# reference picture. Keep it exact and unchanging between runs - a
# word altered is a different child.
#
# It is twenty of the fifty words that fit, and worth every one.
WHO = ("Nik, a 3 year old boy, brown skin, dark brown hair, blue "
       "t-shirt, yellow dungarees")

# The name at the front of WHO, and the words that stand in for it.
#
# WHO goes into a drawing only when the shot is about him. It was
# going into every one, including "Close up of the kitten meowing" -
# so a description of a boy was in front of the model while it drew a
# cat, and it drew a cat with a boy in it.
LEAD = WHO.split(",")[0].strip()

# Whole words only. "he " as a substring is inside "the", so every
# line with the word "the" in it counted as a line about the boy - and
# "Wide shot of the puppy" was given a boy's description to draw.
STANDS_FOR = r"\b(he|his|him|himself|she|her|hers|herself|they|them)\b"

# Where the camera stands, said first and said plainly.
#
# "wide shot, full body in frame" sat third in the prompt and was
# ignored: a drawing came back as a head and two hands, no body, no
# legs, and the puppy half out of the bottom edge. Naming the crops in
# DRAW_NEGATIVE did not win either.
#
# It leads now. A drawing model reads the front of a prompt hardest,
# and the front is where a shot type belongs - it is the first thing a
# director says, before what anybody does.
# "wide shot" and "full body" both came back as a head and shoulders.
# "long shot" is the term a picture model was trained on, and the
# distance has to be said as a distance.
DRAW_FRAMING = ("long shot, full body, standing far from the camera, "
                "head to feet visible")

# Unless the line asked for a close up, in which case fighting it with
# "full body" gives the model two contradictory instructions and it
# picks one. A close up of an animal is exactly what a "Meow, meow,
# meow!" line wants.
DRAW_CLOSE = "close up, filling the frame"


# The words a picture model files a head-and-shoulders under. Two
# drawings came back as exactly this after being asked twice for a full
# body - so they are pushed away, but only on the shots that are not
# meant to be close ups. Putting them on a close up is telling it to do
# and not do the same thing.
NOT_CLOSE = ("close-up, closeup, portrait, headshot, bust, upper body, "
             "waist up, head and shoulders, cropped legs, cropped feet")


def a_close_up(line):
    """Did the line itself ask for a close up?"""

    return line.strip().lower().startswith(("close up", "close-up"))


def framing_for(line):
    """The shot type the line asked for, or the default wide one."""

    return DRAW_CLOSE if a_close_up(line) else DRAW_FRAMING


def draw_negative_for(line):
    """What to push away, given the shot type the line asked for."""

    return (DRAW_NEGATIVE if a_close_up(line)
            else f"{DRAW_NEGATIVE}, {NOT_CLOSE}")

DRAW_STYLE = "3D Pixar CGI, cinematic lighting"

DRAW_NEGATIVE = ("blurry, low detail, deformed face, extra limbs, "
                 "watermark, text, cropped head, extreme close up, 2d, "
                 "flat colours, cel shading, black outlines, "
                 # The drawing that started all this: a well lit boy
                 # against a flat black nothing. Named, because the
                 # reference picture is cropped tight on purpose now
                 # and its own plain background comes with it.
                 "plain background, black background, dark background, "
                 "studio backdrop, empty background, indoors, "
                 # The legs, the hand and the head, each cut by an edge
                 # of the frame in one drawing.
                 "cut off legs, cropped feet, cropped hands, cropped "
                 "arms, subject at the edge of the frame, "
                 # A boy with cat's ears, from a line about a kitten.
                 "hybrid creature, animal ears on a person, "
                 "human face on an animal, merged characters, "
                 # Three of the duckling scenes came back against a
                 # flat grey nothing where the sky should be.
                 "grey background, blank grey sky, overcast empty sky")


# Where the whole video happens.
#
# This was missing entirely, and it showed: a drawing came back with
# Nik standing well lit and well framed against a flat black nothing.
# Nobody had told it where he was. The script line mentions flowers and
# grass in passing, but as details attached to a puppy - not as a
# place, and a drawing model given no place draws a studio backdrop.
#
# It also has to be the SAME place in all sixteen, for the same reason
# the character does. A boy who is the same boy in a different world
# each time is not a series, it is sixteen unrelated pictures.
#
# Keep it short. It shares seventy-seven tokens with everything else.
PLACE = "sunny meadow, blue sky"

# Words that begin a clause about the background rather than about
# somebody. Nothing but the first word of a clause is looked at, so
# "the puppy bouncing on the grass" is kept and "grass rippling gently"
# is not.
#
# These are exactly the words the brief asks a script writer to use for
# background life. Add to them for a show set somewhere else - snow,
# waves, sand - and take one out the moment it becomes a character. A
# show whose star is a bird must not have "bird" in this list.
SCENERY = ("flower", "butterfl", "leaf", "leaves", "cloud", "grass",
           "petal", "breeze", "wind", "sunlight", "dust")


# What "animated" is supposed to mean.
#
# The clips came back at a movement score of 3.67 against the 15 to 20
# the children's channels run at, and the actions in the script were
# not the whole of it: a video model will happily animate one arm and
# leave everything else standing perfectly still, because nothing asked
# it not to.
#
# This is written as a description rather than as a complaint - "nothing
# is frozen" is a negation, and the last time this notebook told a model
# to be unlike a still picture, the face melted at two seconds.
MOTION = (
    "everyone in the shot is moving at once - the boy and every animal "
    "with him - at a calm unhurried pace, his mouth opening and "
    "closing as he sings along, smooth continuous movement all the way "
    "through, natural secondary motion in his hair, his clothes and "
    "the grass, fluid character animation"
)

# Where the camera stands, and it is not optional.
#
# The first Wan clip came back with the boy filling the frame and the
# top of his head cut off, and by two seconds he had walked out of the
# right of the picture and left an empty meadow. Neither is a fault of
# the model: nothing in the prompt said how far away to stand or that
# he had to stay, so it chose - and a video model's idea of a shot is
# whatever its training data did most, which is close.
#
# This sits second in the prompt, straight after the action, because
# the front of a prompt is what is read hardest.
FRAMING = (
    "medium wide shot, the whole of him in frame from his hair to his "
    "shoes, standing in the middle of the picture with space above his "
    "head, he stays fully in frame for the whole shot, he faces the "
    "camera"
)

# The look. These words matter more than any other line in this cell.
#
# "children's cartoon" was in here, and that is precisely the phrase
# that fetched back flat ChuChu-TV shading - it is what most of the
# training data called itself. Naming the craft instead of the audience
# is what pulls it towards the render you actually want, and the flat
# look is pushed away in the negative prompt rather than merely not
# asked for.
STYLE = (
    "3D rendered CGI animation, Pixar and DreamWorks quality, "
    "physically based rendering, ray traced soft shadows, ambient "
    "occlusion, subsurface scattering on skin, glossy specular "
    "highlights, rounded three dimensional forms, deep background with "
    "real distance, volumetric sunlight, shallow depth of field, "
    "cinematic lighting, highly detailed"
)

# "static image, no movement" used to be in here, and it was the single
# worst line in this notebook: it tells the model to look UNLIKE the
# still it was given, which is the one thing it must not do. The face
# melted at two seconds and the scene was gone by four.
#
# Ask for the faults you do not want. Do not ask it to leave the picture.
NEGATIVE = (
    "worst quality, blurry, distorted, deformed face, melting face, "
    "morphing, warping, extra limbs, watermark, text, subtitles, "
    "camera zoom, camera pan, characters vanishing, "
    # The flat look has to be pushed away, not merely left unasked for.
    "flat shading, simple cartoon, low detail, cheap animation, "
    "mobile game graphics, plain empty background, stiff pose, "
    # The last clip came back as flat vector art with hard black
    # outlines. These are that look, named.
    "2d, vector art, flat colours, cel shading, black outlines, "
    "line art, clip art, storybook illustration, flash animation, "
    # The first Wan clip cropped his head and then walked him out of
    # the frame. Both are named here as faults, which is the only way
    # to push them away.
    "cropped head, head cut off, top of head out of frame, "
    "extreme close up, face filling the frame, subject too close, "
    "character walking out of frame, character leaving the shot, "
    "empty frame, empty background with nobody in it, "
    # The first Wan clip moved the boy and left the puppy and the
    # kitten standing like ornaments, and it moved him too fast. Both
    # are named - narrowly. "static image" is NOT in here and must not
    # be: that phrase asks the model to be unlike the picture it was
    # given, and it is what melted a face at two seconds.
    "frozen animals, motionless puppy, only one character moving, "
    "jerky motion, frantic movement, sped up, time lapse, "
    # Four seconds in, the boy had turned magenta and his legs bent
    # where a leg does not bend.
    "colour shift, oversaturated, magenta tint, purple tint, "
    "distorted limbs, bending the wrong way, rubbery arms, extra joints"
)

# Seconds per picture when there is no song to divide up.
FALLBACK_SECONDS = 5.0

FPS = 24

# A reference picture of Nik. Off, and here is why.
#
# The idea was right and I could not make it work, and two goes at it
# both failed the same way, from opposite directions.
#
# IP-Adapter carries what it is shown into the picture it draws. What
# it cannot do is say WHICH character in that picture it applies to -
# there is no such control. With Nik on his own in the frame it is
# fine. With Nik and a kitten it is not:
#
#   ip-adapter-plus      put his yellow dungarees, his blue t-shirt and
#                        his blue sneakers on a kitten standing upright
#                        on two legs, and dressed the duckling to match.
#
#   ip-adapter-plus-face put his FACE on the kitten - a boy's head on a
#                        cat's body with a tail, on all fours.
#
# Weakening it does not fix that, it only makes the leak fainter. The
# leak is the mechanism.
#
# So he is described rather than copied. WHO below says who he is in
# words, and words attach to the noun they are next to: "Nik, a chubby
# 3 year old boy ... blue t-shirt, yellow dungarees" cannot be applied
# to a cat by accident, because the cat is a different noun.
#
# The scenes are still drawn by SDXL first and animated from the
# drawing - that part was never the problem and it is what keeps the
# framing right.
#
# Set this to "nik" to turn the reference back on. It is worth having
# for shots with nobody else in them, and it will dress the animals in
# his clothes in every shot that has one.
REFERENCE_NAME = ""

# How hard the reference pulls, 0 to 1, when there is one.
LIKENESS = 0.4

# Which part of him it carries. "plus" is the whole picture, "plus-face"
# is the face. Both leak onto the other characters; see above.
ADAPTER = "ip-adapter-plus-face_sdxl_vit-h.safetensors"

# What draws those scenes. SDXL is free to use commercially, runs in
# about ten seconds a picture, and takes the reference through
# IP-Adapter, which is Apache 2.0.
DRAW_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"

# Which model. Left blank, the machine decides: Wan on a card with room
# for it, the small LTX otherwise. Set it to "big" or "small" to insist.
FORCE_MODEL = ""

# How good, against how long. One setting, three answers.
#
#   "draft"   832x480,  20 steps - a quick look. Soft at 1080p, and
#                                  the fastest way to find out whether
#                                  a script and a song work together.
#   "good"    1024x576, 30 steps - the default.
#   "best"    1280x704, 30 steps - the size the 5B was trained at.
#                                  Sharpest, and roughly three times
#                                  the time of "good".
#
# Size matters here more than it looks. The finished video is 1080p, so
# 832x480 has to be stretched 2.3 times to reach it and 1024x576 only
# 1.9 - and that stretch is most of why the finished file has looked
# softer than the clip it was made from.
QUALITY = "good"

# Speed instead of movement.
#
# False is Wan 2.2, the model that actually moves things. True is LTX
# 13B distilled: eight steps instead of thirty, roughly four times
# faster, and the price is paid twice over. It runs at guidance_scale
# = 1.0, where there is no classifier-free guidance at all - the
# NEGATIVE prompt below is not read and the positive one steers only
# weakly - and holding the picture still is the thing it is best at,
# which is the complaint about every video this notebook has made.
#
# Use it to check that a script and a song line up before spending an
# hour on the real thing. Do not use it for the video you upload.
FAST_MODEL = False

# How long one generated clip is, before looping.
#
# This is the number that decides whether the video looks right, and
# shorter is safer. Everything in the picture drifts as the clip goes
# on, and the smallest things go first: over three seconds the
# butterflies had melted by one second and the animals' faces by about
# one and a half. Two seconds stays clean.
#
# A picture that has to be on screen longer is not given a longer clip.
# The clip is played forwards, then backwards, then forwards again, for
# as long as it is needed - a sway reads as continuous that way, and the
# model is never asked for more than it can do.
#
# Raise it for more movement in one go, at the cost of more drifting.
# The honest fix for a long song is more pictures, not longer clips.
CLIP_SECONDS = 2.0

# How long one shot holds the screen before the edit cuts away.
#
# This is the number that makes a video look made rather than assembled.
# Children's channels cut every two to four seconds and land every cut
# on the beat of the song; a shot that sits still for ten seconds reads
# as a slideshow however good the picture is.
#
# It also happens to be the fix for the model: cut before three seconds
# and a clip is never on screen long enough to drift.
SHOT_SECONDS = 2.8

# The bottom of every frame is thrown away, as a percentage of height.
# LTX 0.9.8 distilled stamps a line of garbled caption text there. No
# negative prompt removes it - with that model the negative prompt is
# not read at all - and 14 was not quite enough: it came back faintly on
# the next clip, sitting a little higher.
CROP_BOTTOM = 17

# The finished video. 1080p is what YouTube treats as HD.
OUTPUT_WIDTH, OUTPUT_HEIGHT = 1920, 1080

# Also write a vertical cut for Shorts.
MAKE_SHORT = True

# Stop after the drawings, and show all of them at once.
#
# Drawing every scene takes about ten minutes. Animating them takes
# three and a half hours. Looking at all of them before the second part
# starts is the cheapest check in this notebook by a very long way -
# and one sheet with every scene on it answers more in one go than
# forty rounds of sending one clip each ever will.
#
# The drawings are kept, so setting this to False and running again
# costs nothing: it picks up where it stopped.
CHECK_DRAWINGS = True


# Make ONE clip from the first picture and stop.
#
# Worth doing before every real run, and certainly before a new
# character or a new prompt: two minutes of GPU tells you what the model
# does with your picture, instead of finding out eleven clips later.
# The song is ignored while this is on.
TEST_ONE_PICTURE = False


# The run stops when there are too few pictures for the song, because
# the clips would have to be slowed past the point of looking right
# and a paid GPU should not be spent finding that out. Set this True
# to go ahead anyway.
ALLOW_SLOW_CLIPS = False



# ======================================================================
# The card decides the quality, not you
# ======================================================================

# Not having one is fine here. Colab charges for a GPU from the moment
# it connects, whatever you run on it, so everything that does not need
# one is done first - on a CPU runtime, which costs nothing. Only when
# the files are known to be right is a GPU worth connecting.
HAS_GPU = torch.cuda.is_available()

# Wan's pipeline calls ftfy.fix_text on every prompt, and diffusers
# imports ftfy only if it is already installed - so a runtime without it
# gets no warning, loads a 31GB model, and then dies on the first clip
# with "name 'ftfy' is not defined". Cell 1 installs it; this is here
# because that is a very long way to travel to find out.
#
# It has to happen before diffusers is first imported, because that
# import is what decides whether ftfy is available. Nothing above this
# line imports it.
if HAS_GPU:

    import importlib.util

    if importlib.util.find_spec("ftfy") is None:

        print("Installing ftfy - Wan reads every prompt through it.")

        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "ftfy"],
            check=False,
        )

if HAS_GPU:
    CARD = torch.cuda.get_device_name(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    # Assume the card the settings would be chosen for, so the advice
    # about picture counts is the advice you will actually get.
    CARD, VRAM = "none yet - checking your files first", 24.0

# Compute capability 8.0 (Ampere) or newer is where bfloat16 is real.
# Do not ask torch.cuda.is_bf16_supported() - it says True on a T4,
# because torch emulates bfloat16 in software rather than refusing, and
# that emulation is slower than it is worth.
MAJOR = torch.cuda.get_device_capability()[0] if HAS_GPU else 8

DTYPE = torch.bfloat16 if MAJOR >= 8 else torch.float16

# Ordinary RAM matters as much as the card here, and is the thing that
# actually killed the earlier attempts. The 9GB text encoder is unpacked
# in RAM before it ever reaches the GPU, so a big card on a small-RAM
# runtime still dies. Squeeze the text encoder to 8-bit whenever either
# one is short, not just when the card is.
try:
    import psutil

    RAM = psutil.virtual_memory().total / 1e9

except ImportError:
    # Colab ships psutil, but a notebook that dies on a missing helper
    # before it has even looked at your files is no use to anyone.
    import os

    RAM = (
        os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    )

ROOMY_CARD = VRAM >= 20
ROOMY_RAM = RAM >= 20

# What the card decides is whether the 9GB text encoder has to be
# squeezed into 8-bit. It does NOT decide the video size: that is set by
# what the model was trained on, not by how much room there is.
QUANTISE = not (ROOMY_CARD and ROOMY_RAM)

# ---------------------------------------------------------- the model

# LTX comes in two sizes and the difference is not subtle.
#
# The 2B is the original. It runs anywhere and it drifts - the smallest
# things in a picture melt within a second or two.
#
# The 13B is distilled, which means it does in 8 steps what the 2B needs
# 50 for, and it takes image_cond_noise_scale - a setting the 2B's
# pipeline does not even have. At 0.0 no noise is added to your picture
# before it starts, which is precisely the thing that lets a clip wander
# away from it. It needs a 24GB card and room in ordinary RAM to be
# streamed through.
BIG = (VRAM >= 20 and RAM >= 30) if not FORCE_MODEL else FORCE_MODEL == "big"

if BIG and not FAST_MODEL:

    # Wan 2.2 TI2V-5B. Apache 2.0 - free commercially, with no revenue
    # ceiling attached to it the way LTX has one.
    #
    # The reason to move is motion. Every video this notebook has made
    # was LTX, and the complaint about every one of them was the same:
    # not enough happens. That is not a prompting fault that better
    # words would fix. LTX 13B is built to hold a picture steady and it
    # is very good at it - ask it for a parade and it gives you a
    # photograph of one.
    #
    # Wan is trained on movement, and it is a third of the size, so it
    # needs none of the fp8 and leaf-level machinery below: five
    # billion parameters for the video and an 11GB text encoder,
    # offloaded a whole component at a time, fit a 24GB card easily.
    FAMILY = "wan"

    MODEL = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"

    STEPS = 30
    TIMESTEPS = None

    # Real guidance, so the NEGATIVE prompt is read. 5.0 is the number
    # the 5B was tuned at.
    GUIDANCE = 5.0

    # How much of the schedule goes on the big shapes rather than the
    # last fine details. Set below, once the size is known - because it
    # is not one number. Wan is tuned at 5.0 for 720p and 3.0 for 480p,
    # and using the 720p figure on a 480p clip spends the schedule on
    # shapes and leaves too little for anything to settle: the picture
    # drifts magenta and the limbs bend where a boy has no joints.
    FLOW_SHIFT = 5.0

    # Divisible by 32 both ways, which the Wan VAE requires: it
    # compresses 16x in space and the patch size doubles that again.
    WIDTH, HEIGHT = 1024, 576

    if QUALITY == "draft":
        WIDTH, HEIGHT = 832, 480

        # Was 20, and 20 was too few to carry guidance 5.0: the colour
        # ran and the arms warped. A draft is meant to be smaller, not
        # to tell a different story about what the model does.
        STEPS = 25

    elif QUALITY == "best":
        WIDTH, HEIGHT = 1280, 704

    # Only the 480p draft takes the lower figure.
    #
    # The temptation was to give 576 the 480p number too, on the
    # grounds that it is nearer 480 than 720. The evidence says
    # otherwise: 1024x576 at 5.0 and thirty steps produced a whole
    # video with no colour drift in it at all, and the clip that went
    # magenta was 832x480 at twenty steps. Changing a setting that is
    # working, because a rule of thumb says it might be wrong, is how
    # you lose a good result chasing a bad one.
    FLOW_SHIFT = 3.0 if HEIGHT <= 480 else 5.0

    QUANTISE = False

elif BIG:

    FAMILY = "ltx"

    MODEL = "Lightricks/LTX-Video-0.9.8-13B-distilled"

    # Distilled: few steps, and the exact schedule it was distilled
    # for. These are not tuneable, they came with the model - and
    # guidance 1.0 means the negative prompt is not read at all.
    STEPS = 8
    TIMESTEPS = [1000, 993, 987, 981, 975, 909, 725, 0.03]
    GUIDANCE = 1.0
    FLOW_SHIFT = None

    WIDTH, HEIGHT = 960, 544

    # It offloads itself, layer by layer. An 8-bit text encoder pinned
    # to the card by device_map would only get in the way of that.
    QUANTISE = False

else:

    FAMILY = "ltx"

    MODEL = "Lightricks/LTX-Video"

    STEPS = 50
    TIMESTEPS = None
    GUIDANCE = 3.0
    FLOW_SHIFT = None

    # This one was trained near 704x480 - about 338,000 pixels. Asking
    # for 1024x576 is 75% more, and it showed: the picture came apart
    # after two seconds. 768x448 is what it knows, in 16:9.
    WIDTH, HEIGHT = 768, 448

# Only LTX 0.9.8 distilled stamps that band of caption text along the
# bottom. Wan does not, and throwing away 17% of a picture with nothing
# wrong with it is just throwing away picture.
if FAMILY == "wan":
    CROP_BOTTOM = 0

# The loop was itself a reason these videos read as static: a two
# second clip played forwards, then backwards, then forwards again is
# a wobble - the parade walks off and then walks back in. LTX could
# not be trusted past two seconds so there was no choice about it.
# Wan holds together, so give each shot one unbroken clip and let the
# movement go one way.
#
# Longer than the shot needs, and deliberately so. A model given "he
# claps three times" and three seconds does three claps in three
# seconds, which is frantic - and the first clip was. Given four, the
# same three claps are spread over four, and the edit shows the first
# 2.8 of them. Pace is set by how much room the action is given, not by
# any setting called speed.
if FAMILY == "wan":
    CLIP_SECONDS = max(CLIP_SECONDS, SHOT_SECONDS + 1.2)

# Frames the model is asked for. (frames - 1) has to divide by the
# VAE's temporal stride - 4 for Wan, 8 for LTX - and the rounding goes
# to the nearest: rounding down turned a 3.0s clip into a 2.7s one.
STRIDE = 4 if FAMILY == "wan" else 8

MAX_FRAMES = max(25, round((CLIP_SECONDS * FPS - 1) / STRIDE) * STRIDE + 1)

print(f"Notebook    : {BUILD}")
print(f"GPU         : {CARD}" + (f" ({VRAM:.0f}GB)" if HAS_GPU else ""))
print(f"System RAM  : {RAM:.0f}GB")
print(f"Precision   : {'bfloat16' if MAJOR >= 8 else 'float16'}"
      f"{', text encoder in 8-bit' if QUANTISE else ''}")
print(f"Model       : {MODEL.split('/')[-1]}"
      f" ({'5B' if FAMILY == 'wan' else '13B' if BIG else '2B'}"
      f", {STEPS} steps)")
print(f"Guidance    : {GUIDANCE}"
      + ("  - the prompt pulls, the negative prompt is read"
         if GUIDANCE > 1.0 else
         "  - NO guidance: the negative prompt is ignored and the\n"
         "              style words steer only weakly. "
         "FAST_MODEL = False to change that."))
print(f"Generated at: {WIDTH}x{HEIGHT}, "
      f"{MAX_FRAMES} frames ({MAX_FRAMES / FPS:.1f}s"
      + (", one unbroken take per shot" if FAMILY == "wan"
         else ", looped to fill each shot") + ")")
print(f"Video out   : {OUTPUT_WIDTH}x{OUTPUT_HEIGHT}")


# ======================================================================
# Your files
# ======================================================================

try:
    from google.colab import files as colab_files
    from google.colab import drive as colab_drive

except ImportError:
    colab_files = colab_drive = None      # not in Colab


def mount_drive():
    """
    Connect Drive, and say plainly if it will not.

    Returns True when Drive is there. Not connecting is no longer fatal:
    the session's own disk works perfectly well, and asking someone to
    fight their browser settings before they can see a single clip is
    not a reasonable thing to do.
    """

    if colab_drive is None or Path(DRIVE).exists():
        return True

    try:
        colab_drive.mount("/content/drive")
        return True

    except Exception as trouble:

        # "credential propagation was unsuccessful" is the usual one,
        # and it is the browser refusing the sign-in popup - nothing to
        # do with Drive, the files, or this notebook.
        print(
            f"\n  Google Drive would not connect: {trouble}\n"
            "\n  That is the browser refusing the sign-in popup. If you "
            "want to use Drive:\n"
            "    1. Allow third-party cookies for "
            "colab.research.google.com - in Chrome, the\n"
            "       eye or padlock icon in the address bar > Cookies - "
            "and run this cell again.\n"
            "    2. An ordinary window, not Incognito or a guest "
            "profile.\n"
            "    3. Runtime > Disconnect and delete runtime, then "
            "reconnect.\n"
            "\n  Carrying on without it. Upload your files below and "
            "download the video at\n  the end - it works exactly the "
            "same, it just does not survive the session."
        )

        return False


ON_DRIVE = mount_drive()

root = Path(FOLDER) if ON_DRIVE else Path(LOCAL)

INPUT = root / "Input"
OUTPUT = root / "Output"
CLIPS = OUTPUT / "Clips"


def ask_for_files(folder):
    """Take the script, the song and any pictures straight from the PC."""

    folder.mkdir(parents=True, exist_ok=True)

    print("\n  Upload your script (.txt) and your song - and your "
          "pictures if you\n  are using pictures. You can pick them all "
          "at once.\n")

    for name in colab_files.upload():
        shutil.move(name, folder / name)

    print()


if not ON_DRIVE and colab_files is not None:

    if not INPUT.exists() or not any(INPUT.iterdir()):
        ask_for_files(INPUT)


def looks_like_input(folder):
    """A folder with pictures in it is a folder worth offering."""

    try:
        return any(
            item.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp")
            for item in folder.iterdir()
        )
    except OSError:
        return False


def hunt_for_input():
    """
    Where the pictures actually are.

    "No such folder" is a dead end when the folder plainly exists on the
    other machine - Drive may be a different account, or the path may be
    a level off. Looking is more use than complaining, so anything named
    Input with pictures in it counts, and so does a NikStudio folder.
    """

    drive = Path(DRIVE)

    if not drive.exists():
        return []

    found = []

    for depth in ("*", "*/*", "*/*/*", "*/*/*/*"):

        for folder in drive.glob(depth):

            if not folder.is_dir():
                continue

            if folder.name.lower() in ("input", "nikstudio"):
                if looks_like_input(folder):
                    found.append(folder)

    return sorted(set(found))


if not INPUT.exists() and ON_DRIVE:

    print(f"\n  {INPUT} is not there. Looking for it ...")

    candidates = hunt_for_input()

    if len(candidates) == 1:

        INPUT = candidates[0]

        root = INPUT.parent
        OUTPUT = root / "Output"
        CLIPS = OUTPUT / "Clips"

        print(f"  Found your pictures in {INPUT} - using that.")

    elif candidates:

        raise SystemExit(
            f"No such folder: {INPUT}\n\n"
            "These have pictures in them - put the right one in FOLDER "
            "at the top of this cell\n(FOLDER is the folder ABOVE Input):"
            "\n\n"
            + "\n".join(f"    {folder}" for folder in candidates)
        )

    else:

        visible = sorted(
            item.name for item in Path(DRIVE).glob("*") if item.is_dir()
        ) if Path(DRIVE).exists() else []

        raise SystemExit(
            f"No such folder: {INPUT}\n\n"
            "Nothing with pictures in it was found anywhere in this "
            "Drive.\n\n"
            "Two things to check:\n"
            "  1. Colab is mounted on the same Google account your Drive "
            "folder is on.\n"
            "  2. Google Drive on your PC has finished uploading - a "
            "folder that only\n     exists locally is not there yet.\n\n"
            "The top level of the Drive Colab can see:\n\n"
            + ("\n".join(f"    {name}" for name in visible[:30])
               or "    (nothing)")
        )

CLIPS.mkdir(parents=True, exist_ok=True)

def natural_key(path):
    """
    Sort "Scene2" before "Scene10".

    Plain alphabetical order puts "10" before "2", which silently
    shuffles someone's scenes. Numbers in a name are compared as numbers.
    """

    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", path.name)
    ]


PICTURES = sorted(
    (path for path in INPUT.iterdir()
     if path.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp")),
    key=natural_key,
)

# The one picture that is not a scene. Taken out of the list before
# anything else sees it, so it cannot be animated as though it were
# one, and cannot be counted when the run works out whether there are
# enough scenes for the song.
REFERENCE = next(
    (path for path in PICTURES
     if REFERENCE_NAME
     and path.stem.lower().startswith(REFERENCE_NAME.lower())),
    None,
)

if REFERENCE is not None:
    PICTURES = [path for path in PICTURES if path != REFERENCE]

def is_lyrics(path):
    """A lyric sheet is any .txt whose name starts with "lyric"."""

    return path.stem.lower().startswith("lyric")


def lyrics_in(folder):
    """The words of the song, to put on screen. One line a line."""

    sheets = sorted(
        (item for item in folder.iterdir()
         if item.is_file() and item.suffix.lower() == ".txt"
         and is_lyrics(item)),
        key=natural_key,
    )

    if not sheets:
        return None, []

    lines = [
        line.strip()
        for line in sheets[0].read_text(encoding="utf-8").splitlines()
    ]

    return sheets[0], [l for l in lines if l and not l.startswith("#")]


def script_in(folder):
    """
    The script, whatever it ended up being called.

    Not `folder / "script.txt"`. Windows hides extensions, so renaming
    a file to "script.txt" in Explorer quietly produces script.txt.txt;
    Drive is case sensitive, so Script.txt is a different file; and
    people name things scenes.txt. There is no other reason for a .txt
    to be in here, so any of them is the script - preferring one that
    is at least called script.
    """

    texts = sorted(
        (item for item in folder.iterdir()
         if item.is_file() and item.suffix.lower() == ".txt"
         and not is_lyrics(item)),
        key=natural_key,
    )

    named = [t for t in texts if t.stem.lower().startswith("script")]

    return (named or texts)[0] if texts else None


SCRIPT = script_in(INPUT)

LYRIC_SHEET, LYRICS = lyrics_in(INPUT)

SONG = next(
    (
        path for path in sorted(INPUT.iterdir(), key=natural_key)
        if path.suffix.lower() in (".mp3", ".wav", ".m4a", ".aac", ".ogg")
    ),
    None,
)


def seconds_of(media):
    """How long an audio or video file runs, in seconds."""

    probe = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(media),
        ],
        capture_output=True,
        text=True,
    )

    try:
        return float(probe.stdout.strip())
    except ValueError:
        return 0.0


def contact_sheet(pictures, target, across=6, wide=380):
    """
    Every drawing on one page, numbered.

    One image that can be looked at in a second and sent in a message.
    Checking a video by asking for one clip at a time is how a fault
    that is in all forty-four of them takes forty rounds to find.
    """

    if not pictures:
        return None

    from PIL import ImageDraw

    try:
        from PIL import ImageFont

        label = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 26
        )

    except Exception:
        # Every Pillow has this one, and a small label beats no label.
        label = None

    with Image.open(pictures[0]) as first:
        tall = round(wide * first.height / first.width)

    down = -(-len(pictures) // across)

    sheet = Image.new("RGB", (across * wide, down * tall), (18, 18, 18))

    pen = ImageDraw.Draw(sheet)

    for number, path in enumerate(pictures):

        with Image.open(path) as picture:
            small = picture.convert("RGB").resize(
                (wide, tall), Image.LANCZOS
            )

        left = (number % across) * wide
        top = (number // across) * tall

        sheet.paste(small, (left, top))

        name = path.stem

        pen.rectangle(
            (left, top, left + 12 + 11 * len(name), top + 34),
            fill=(0, 0, 0),
        )

        pen.text((left + 6, top + 4), name, fill=(255, 220, 90),
                 font=label)

    sheet.save(target, quality=80)

    return target


def motion_of(video, seconds=40.0):
    """
    How much actually moves, as one number.

    Every frame is subtracted from the one before and what is left is
    averaged. A photograph scores 0. The videos this notebook made with
    LTX scored between 3 and 10; the children's channels it is aimed at
    score 15 to 20.

    It exists because a number can be pasted into a message and a video
    often cannot.
    """

    reading = subprocess.run(
        [
            "ffmpeg", "-v", "error", "-t", f"{seconds:.1f}",
            "-i", str(video),
            "-vf", ("scale=160:90,format=gray,"
                    "tblend=all_mode=difference,"
                    "signalstats,metadata=print:file=-"),
            "-f", "null", os.devnull,
        ],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    scores = [
        float(line.rsplit("=", 1)[-1])
        for line in (reading.stdout or "").splitlines()
        if "signalstats.YAVG" in line
    ]

    # The first frame has nothing before it, so its difference is the
    # whole picture. Dropping it stops a one-off from doubling a short
    # clip's score.
    scores = scores[1:] or scores

    return sum(scores) / len(scores) if scores else 0.0


# ----------------------------------------------------------- the shots

# A shot is one clip: a name, the words the model is given, and the
# picture it starts from if there is one. Everything downstream works on
# these, so written scenes and pictures go through the same pipeline.

# SDXL reads 77 tokens, which is about 52 words once the punctuation
# is counted. WHO, PLACE, the framing and the style take a fixed share
# of that, and what is left is what a script line may spend.
ROOM = max(8, 52 - len(f"{WHO} {PLACE} {DRAW_FRAMING} {DRAW_STYLE}".split()))


def scenery(clause):
    """
    Is this clause the weather rather than somebody?

    A background clause names the background first - "flowers nodding",
    "butterflies drifting past", "grass rippling gently" - which is how
    the brief asks for them. A clause about a character names the
    character first, and "the puppy bouncing on the grass beside him"
    mentions grass without being about grass.

    So the FIRST word decides, not any word. That keeps this working
    for a show whose cast this notebook has never heard of: anything it
    does not recognise is assumed to be somebody, which is the safe way
    round to be wrong.
    """

    first = clause.strip().lower()

    for article in ("the ", "a ", "an "):
        if first.startswith(article):
            first = first[len(article):]

    return first.startswith(SCENERY)


# "Split of Scene07 and Scene11" - two or three finished clips, side by
# side in one frame.
#
# Every drawing has one character in it, because that is the only way a
# picture model draws them without merging them into each other. This
# is how they get to share a frame anyway: not by asking the model for
# something it cannot do, but by cutting up clips that already exist.
#
# No GPU at all. It costs one ffmpeg call and cannot produce a hybrid,
# because nothing is being generated.
SPLIT_OF = re.compile(r"^\s*split\s+of\b", re.I)

SCENE_NAME = re.compile(r"\bScene\d+\b", re.I)


def split_in(line):
    """The scenes a "Split of ..." line asks to put side by side."""

    if not SPLIT_OF.match(line):
        return []

    return [found.group(0) for found in SCENE_NAME.finditer(line)]


def about_the_lead(line):
    """
    WHO, but only when the shot is actually about him.

    "Close up of the kitten meowing twice" came back as a boy with
    cat's ears on his head, because a description of a boy was sitting
    in the prompt while the model drew a cat. There was no boy in the
    line at all - only in the words underneath it.

    Nothing in here is about this show: the name comes from WHO and the
    pronouns are just English.
    """

    low = line.lower()

    if LEAD.lower() in low or re.search(STANDS_FOR, low):
        return WHO

    return ""


def still_of(line):
    """
    The part of a script line a drawing needs, inside its budget.

    The first clause always - that is the action. After that, only
    clauses that mention somebody. "butterflies drifting past" and "the
    camera does not move" are for the video; a still has no camera and
    nothing drifts in it.

    Then trimmed from the end until it fits. Losing the third animal
    from a crowded line costs less than losing the style words off the
    end, which is what silently happened before - SDXL cuts at 77
    tokens without stopping and without saying which words went.
    """

    clauses = line.split(", ")

    kept = [clauses[0]]

    for clause in clauses[1:]:

        if "camera" in clause.lower():
            continue

        if not scenery(clause):
            kept.append(clause)

    while len(kept) > 1 and len(" ".join(kept).split()) > ROOM:
        kept.pop()

    return ", ".join(kept)


def from_script(path):

    lines = [
        line.strip()
        for line in path.read_text(encoding="utf-8").splitlines()
    ]

    # Blank lines space a script out; a # is how you park a scene
    # without deleting it.
    lines = [line for line in lines if line and not line.startswith("#")]

    return [
        {
            "name": f"Scene{number:02d}",
            "written": line,
            # The ACTION leads. It used to sit in the middle, after
            # sixty words describing the character, and a clip came
            # back of a boy standing still in an empty field - no
            # puppy, no flowers, none of what the line asked for.
            #
            # A video model weighs the front of a prompt hardest, and
            # this one weighs everything weakly, so burying the one
            # thing that is meant to happen behind a costume
            # description is throwing it away. Framing second - a shot
            # with his head cut off is not saved by anything that comes
            # after it - then the character, then the style.
            #
            # Full stops between the three, or the action runs straight
            # into the description and they are read as one clause.
            "prompt": ". ".join(
                part.rstrip(" .,")
                for part in (line, MOTION, PLACE, FRAMING, CHARACTER,
                             STYLE)
            ) + ".",
            # What SDXL is given instead, when there is a reference
            # picture to draw from. Short enough to survive 77 tokens.
            #
            # Only what a still picture needs.
            #
            # A script line is four clauses: the action, who else is
            # in it, a piece of background life, and the camera. A
            # drawing has no camera and does not need drifting
            # butterflies - they are for the video, where things move.
            # Dropping both pays for WHO, and WHO is what stops a cat
            # turning up in his dungarees.
            "drawing": ". ".join(
                part.rstrip(" .,")
                for part in (framing_for(line), still_of(line),
                             about_the_lead(line), PLACE, DRAW_STYLE)
                if part
            ) + ".",
            "drawing_negative": draw_negative_for(line),
            "split": split_in(line),
            "picture": None,
        }
        for number, line in enumerate(lines, start=1)
    ]


def from_pictures(paths):

    return [
        {
            "name": path.stem,
            "written": "",
            "prompt": PROMPTS.get(path.name, PROMPT),
            "split": [],
            "picture": path,
        }
        for path in paths
    ]


if SCRIPT is not None:

    SHOTS = from_script(SCRIPT)

    if not SHOTS:
        raise SystemExit(
            f"{SCRIPT} has no scenes in it. Write one a line."
        )

    WRITTEN = True

    print(f"\nFrom       : {SCRIPT.name}, {len(SHOTS)} scene(s)")

    if PICTURES:
        print(f"             ({len(PICTURES)} picture(s) in the folder are "
              "being ignored - delete script.txt to use them instead)")

elif PICTURES:

    SHOTS = from_pictures(PICTURES)

    WRITTEN = False

    print(f"\nFrom       : {len(SHOTS)} picture(s)")

else:

    inside = sorted(item.name for item in INPUT.iterdir())

    raise SystemExit(
        f"Nothing to work from in {INPUT}.\n\n"
        "Put your pictures in there, or a script.txt with one scene a "
        "line.\n\n"
        "What is in there now:\n\n"
        + ("\n".join(f"    {name}" for name in inside[:40])
           or "    (nothing)")
    )


# Counted before a test run throws them away.
#
# A test run drops the song and the words, which is right - what comes
# back should be the model's work and nothing else. But it also
# silently dropped the one line that says whether the script and the
# song line up, so the cheapest check in the notebook stopped
# answering the most important question in it.
SCENES_WRITTEN = len(SHOTS)

LINES_SUNG = len(LYRICS)

if TEST_ONE_PICTURE:

    SHOTS = SHOTS[:1]

    # Nothing is laid over a test clip. What comes back is the model's
    # work and nothing else - which the lyrics were quietly breaking:
    # the first test came back with a line of the song burned across
    # it, which is not what a test of the model is for.
    SONG = None

    LYRICS = []

    SONG_SECONDS = 0.0

    SHARE = MAX_FRAMES / FPS

    print(f"TEST       : one clip only, {SHARE:.1f}s. The song is ignored.")

elif SONG:
    SONG_SECONDS = seconds_of(SONG)
    SHARE = SONG_SECONDS / len(SHOTS)
    print(f"Song       : {SONG.name} ({SONG_SECONDS:.1f}s)")

else:
    SONG_SECONDS = 0.0
    SHARE = FALLBACK_SECONDS
    print("Song       : none found - using "
          f"{FALLBACK_SECONDS:.0f}s per shot")

if REFERENCE is not None and WRITTEN:

    print(f"Nik        : {REFERENCE.name} - every scene is drawn from "
          "him first")

    # A reference with a whole scene in it is copied as a whole scene.
    #
    # The first sixteen came back with the reference's rainbow, its
    # flowers and its seated pose in every one, whatever the line said,
    # because IP-Adapter carries what it is shown - and it was shown a
    # scene. A tall, tight picture of him is read as a boy; a wide one
    # is read as a place he is standing in.
    #
    # This is a shape, not a judgement, so it is a note rather than a
    # refusal. But it is the single most useful thing to fix.
    try:
        with Image.open(REFERENCE) as look:
            shape = look.width / max(1, look.height)

    except Exception:
        shape = 1.0

    if shape > 1.25:
        print(f"             Note: it is {shape:.1f} times wider than "
              f"it is tall, which is a scene\n             rather than "
              f"a portrait - and the whole scene is what gets copied "
              f"into\n             every shot. Crop it to Nik and "
              f"little else and the shots stop\n             looking "
              f"like sixteen versions of the same picture.")

elif REFERENCE is not None:
    print(f"Nik        : {REFERENCE.name} is set aside as the reference, "
          "but there is no\n             script to draw - the pictures "
          "are animated as they are")

# One scene per line of the song, or not.
#
# This is the difference between a video of a song and a video with a
# song over it. When the counts match, scene four IS line four: the
# picture shows what is being sung, every time, by construction.
#
# The first full video did not match - sixteen scenes against
# forty-four lines - and it played "Meow, meow, meow!" over a duckling
# and "Quack, quack, quack!" over a puppy. Neither the words nor the
# pictures were wrong. They were just not the same story.
LINE_FOR_LINE = bool(LYRICS) and len(SHOTS) == len(LYRICS)

if LINES_SUNG:

    print(f"Lyrics     : {LYRIC_SHEET.name}, {LINES_SUNG} line(s)"
          + (" - they go on screen" if LYRICS else
             " - not on a test clip, but still counted"))

    if SCENES_WRITTEN == LINES_SUNG:
        print("             One scene per line - every picture shows "
              "what is being sung")

    else:
        print(f"\n  The song has {LINES_SUNG} lines and your script "
              f"has {SCENES_WRITTEN} scenes, so the two\n  do not line up. "
              f"Whatever is on screen when a line is sung is whichever\n"
              f"  scene that moment landed in - which is how the last "
              f"video sang\n  \"Meow, meow, meow!\" over a duckling.\n\n"
              f"  Write {LINES_SUNG} scenes, one for each line of "
              f"{LYRIC_SHEET.name}, and every picture\n  shows what is "
              f"being sung. It is the single biggest thing left.")

print(f"Shots      : {len(SHOTS)} "
      f"({', '.join(shot['name'] for shot in SHOTS)})")

# Not "each holds 7.8s" - the edit cuts every SHOT_SECONDS, so nothing
# is on screen for that long. What that number really is, is each
# shot's share of the song.
print(f"Each gets  : {SHARE:.1f}s of the song, cut into pieces")

# ======================================================================
# Everything that can go wrong, found before the GPU is touched
# ======================================================================
#
# Loading the model takes minutes and a paid GPU is charged for them, so
# nothing here is left to be discovered halfway through the run.

problems = []

print()

for shot in SHOTS:

    if shot["picture"] is None:
        print(f"  {shot['name']:<10} {shot['written'][:56]}")
        continue

    picture_file = shot["picture"]

    try:
        with Image.open(picture_file) as check:
            check.verify()

        with Image.open(picture_file) as check:
            shape = check.size

    except Exception:
        problems.append(
            f"{picture_file.name} will not open. Re-save it as a PNG."
        )
        continue

    print(f"  {picture_file.name:<28} {shape[0]}x{shape[1]}")

# A "Split of ..." line that names a scene which is not there, or is
# itself a split, would fail in the middle of the run - after the GPU
# has been paid for.
NAMED = {shot["name"] for shot in SHOTS}

SPLITS = {shot["name"] for shot in SHOTS if shot["split"]}

for shot in SHOTS:

    if not shot["split"]:
        continue

    if not 2 <= len(shot["split"]) <= 3:
        problems.append(
            f"{shot['name']} is a \"Split of\" line naming "
            f"{len(shot['split'])} scene(s). It has to name two or "
            f"three."
        )
        continue

    for name in shot["split"]:

        if name not in NAMED:
            problems.append(
                f"{shot['name']} asks for {name}, and there is no "
                f"{name} - the script has {len(SHOTS)} lines."
            )

        elif name in SPLITS:
            problems.append(
                f"{shot['name']} asks for {name}, which is itself a "
                f"\"Split of\" line. A split can only be made of "
                f"scenes that were animated."
            )

if SONG and not SONG_SECONDS:
    problems.append(
        f"{SONG.name} cannot be read. Try a plain MP3 or WAV."
    )

# The card is not big enough for the model that moves, and an hour on
# the one that does not is an hour wasted.
#
# This is the same mistake as the FAST_MODEL switch, wearing a
# different hat: the notebook knew, printed it, and carried on anyway
# into a video that was never going to be liked. So it stops instead.
# Only once a GPU is really connected - the free check on a CPU
# runtime assumes the card you are about to turn on.
if HAS_GPU and not BIG and not FORCE_MODEL:

    problems.append(
        f"This is a {CARD} with {RAM:.0f}GB of RAM. It can only run the "
        f"2B model,\n         which is the one that drifts rather than "
        f"moves - the model behind every\n         video so far that "
        f"was not lively enough.\n\n"
        f"         Runtime > Change runtime type > L4 GPU > Save\n"
        f"         then run cell 1, then this cell again. Colab Pro "
        f"gives you L4.\n\n"
        f'         To go ahead on this card anyway, set FORCE_MODEL = '
        f'"small" at the top.'
    )

# The edit cuts every SHOT_SECONDS, so a long song is no longer a
# problem of holding one picture too long. It is a question of variety:
# with more shots than clips, each clip comes back more than once.
#
# Twice or three times is how television works - you return to a setup.
# Eight times over two minutes is the same three seconds again and
# again, and no camera move hides that.
SHOTS_IN_EDIT = max(1, round((SONG_SECONDS or SHARE * len(SHOTS))
                             / SHOT_SECONDS))

TIMES_EACH = SHOTS_IN_EDIT / len(SHOTS)

if TIMES_EACH > 8:

    enough = max(1, round(SHOTS_IN_EDIT / 4))

    complaint = (
        f"{len(SHOTS)} shot(s) against {SHOTS_IN_EDIT} cuts means each "
        f"clip comes back {TIMES_EACH:.0f} times.\n"
        f"       The edit cannot hide that. Use about {enough} shots for "
        f"a {SONG_SECONDS:.0f}s song.\n\n"
        f"       Only trying one out? Set TEST_ONE_PICTURE = True at the "
        f"top - it makes\n       one clip, ignores the song, and takes "
        f"about two minutes. Or ALLOW_SLOW_CLIPS = True\n"
        f"       to go ahead as things stand."
    )

    if ALLOW_SLOW_CLIPS:
        print(f"\n  Warning: {complaint}")
    else:
        problems.append(complaint)

elif TIMES_EACH > 4:
    print(f"\n  Note: {SHOTS_IN_EDIT} cuts from {len(SHOTS)} clips, so "
          f"each comes back about {TIMES_EACH:.0f} times.\n"
          f"        Watchable - the camera move differs each time - but "
          f"more shots would be better.")

elif TIMES_EACH > 1.2:
    print(f"\n  {SHOTS_IN_EDIT} cuts from {len(SHOTS)} clips: each "
          f"clip covers about {TIMES_EACH:.0f} cuts in a row, from a"
          f"\n  different camera move each time.")
    print(f"\n  That reuse is the clearest thing separating this from "
          f"the channels you\n  are aiming at - they never show a shot "
          f"twice. {SHOTS_IN_EDIT} scenes in script.txt\n  and nothing "
          f"repeats at all. It costs GPU time in proportion and "
          f"changes\n  nothing else about the run.")

if problems:

    raise SystemExit(
        "\n\nStopping before the GPU is used:\n\n"
        + "\n".join(f"  -  {problem}" for problem in problems)
        + "\n\nFix these and run this cell again. Nothing has been "
          "charged for."
    )

print("\n  Everything checks out.")

if not HAS_GPU:

    raise SystemExit(
        "\n\nYour files are ready, and this cost nothing - there was no "
        "GPU running.\n\n"
        "Now turn one on and do the work:\n\n"
        "  1. Runtime > Change runtime type > L4 GPU > Save\n"
        "  2. Run cell 1 (the packages), then run this cell again.\n\n"
        f"It will make {len(SHOTS)} clip(s). Nothing else needs "
        "checking."
    )


# ======================================================================
# One picture of him -> a still of every scene he is in
# ======================================================================
#
# This runs before the video model is loaded, and the drawing model is
# thrown away before it is. Two seven gigabyte models on the card at
# once is how a run dies at scene twelve with everything to do again.

SCENES = OUTPUT / "Scenes"

DREW = 0

# Drawn whenever there is a script, reference or not.
#
# The drawing stage was never the problem - it is what gets the framing
# right, because a model that composes pictures is better at it than
# one that moves them. Only the reference leaked, and the reference is
# now optional.
if WRITTEN:

    SCENES.mkdir(parents=True, exist_ok=True)

    SCENE_STAMPS = SCENES / "drawn.json"

    try:
        drawn_before = json.loads(SCENE_STAMPS.read_text(encoding="utf-8"))
    except Exception:
        drawn_before = {}

    TO_DRAW = [shot for shot in SHOTS if not shot["split"]]

    him = (Image.open(REFERENCE).convert("RGB")
           if REFERENCE is not None else None)

    facts = REFERENCE.stat() if REFERENCE is not None else None

    print(f"\nDrawing {len(TO_DRAW)} scene(s)"
          + (f" from {REFERENCE.name}" if REFERENCE is not None
             else " from your script")
          + ". About ten seconds each.")

    # 77 tokens is roughly 50 words once the punctuation is counted,
    # and the fixed parts take 23 of them. This fired on fifteen lines
    # of a script where the only thing actually cut was the word
    # "detailed" and a full stop, which is a warning that costs more
    # attention than the fault does - so the fixed parts are shorter
    # now and the threshold is honest.
    long_lines = [shot["name"] for shot in SHOTS
                  if len(shot["drawing"].split()) > 50]

    if long_lines:
        print(f"  Note: {len(long_lines)} line(s) are long enough that "
              f"SDXL will cut the end off\n        ({', '.join(long_lines[:6])}"
              f"). Shorten them in script.txt if the\n        style comes "
              f"back wrong.")

    from diffusers import StableDiffusionXLPipeline
    from transformers import CLIPImageProcessor, CLIPVisionModelWithProjection

    started = time.time()

    # The one that has to be said out loud.
    #
    # ip-adapter-plus_sdxl_vit-h expects CLIP ViT-H, which is 1280 wide.
    # SDXL ships with ViT-bigG, which is 1664, and diffusers will happily
    # load that one and then fail inside the first step with "mat1 and
    # mat2 shapes cannot be multiplied (514x1664 and 1280x1280)". The
    # numbers in that message are those two encoders.
    #
    # So the encoder is fetched by name and handed over, rather than
    # left to a default that does not match the adapter.
    seeing = {}

    if REFERENCE is not None:

        eyes = CLIPVisionModelWithProjection.from_pretrained(
            "h94/IP-Adapter",
            subfolder="models/image_encoder",
            torch_dtype=DTYPE,
        )

        seeing = {"image_encoder": eyes,
                  "feature_extractor": CLIPImageProcessor()}

    draw = StableDiffusionXLPipeline.from_pretrained(
        DRAW_MODEL,
        torch_dtype=DTYPE,
        variant="fp16",
        use_safetensors=True,
        **seeing,
    )

    # IP-Adapter is what makes the reference mean anything. Without it
    # the picture is decoration; with it, his face and his clothes are
    # carried into a scene he was never in.
    if REFERENCE is not None:

        draw.load_ip_adapter(
            "h94/IP-Adapter",
            subfolder="sdxl_models",
            weight_name=ADAPTER,
        )

        draw.set_ip_adapter_scale(LIKENESS)

    draw.enable_model_cpu_offload()

    for number, shot in enumerate(TO_DRAW, start=1):

        # A side-by-side shot is cut from clips that already exist.
        # There is nothing to draw and nothing to generate.
        scene_file = SCENES / f"{shot['name']}.png"

        wanted = {
            "prompt": shot["drawing"],
            "reference": REFERENCE.name if REFERENCE is not None else "",
            "bytes": facts.st_size if facts else 0,
            "modified": int(facts.st_mtime) if facts else 0,
            "likeness": LIKENESS if REFERENCE is not None else 0,
            "adapter": ADAPTER if REFERENCE is not None else "",
            "model": DRAW_MODEL,
        }

        if (scene_file.exists()
                and drawn_before.get(scene_file.name) == wanted):
            shot["picture"] = scene_file
            continue

        # 1344x768 is one of the sizes SDXL was trained at, and close
        # enough to 16:9 that the crop afterwards takes a hair off the
        # sides. Asking it for 832x480 directly gives a worse picture
        # than asking for a good size and cutting it down.
        asked_for = dict(
            prompt=shot["drawing"],
            negative_prompt=shot["drawing_negative"],
            width=1344,
            height=768,
            num_inference_steps=30,
            guidance_scale=6.0,
            # The same seed for every scene, on purpose. Different
            # words already make different pictures; what a fixed seed
            # fixes is the lottery underneath them, which is where a
            # boy stops being the same boy.
            generator=torch.Generator("cpu").manual_seed(42),
        )

        if him is not None:
            asked_for["ip_adapter_image"] = him

        picture = draw(**asked_for).images[0]

        picture.save(scene_file)

        shot["picture"] = scene_file

        drawn_before[scene_file.name] = wanted

        SCENE_STAMPS.write_text(
            json.dumps(drawn_before, indent=1), encoding="utf-8"
        )

        DREW += 1

        print(f"  [{number}/{len(TO_DRAW)}] {shot['name']}")

    print(f"  {DREW} drawn, {len(TO_DRAW) - DREW} already there, "
          f"in {(time.time() - started) / 60:.1f} min")

    # Gone before the video model arrives. This is the whole reason the
    # drawing happens here rather than inside the clip loop.
    del draw

    gc.collect()

    torch.cuda.empty_cache()

    # Every one of them, on one page.
    #
    # Showing the first drawing was showing one forty-fourth of the
    # answer. A fault that is in all of them - a close up where a wide
    # shot was asked for, the wrong animal on a line, a grey sky -
    # takes one look here and forty rounds the other way.
    SHEET = contact_sheet(
        [shot["picture"] for shot in TO_DRAW], OUTPUT / "Scenes.jpg"
    )

    if SHEET:
        print(f"  All of them: {SHEET} "
              f"({SHEET.stat().st_size / 1e6:.1f}MB - small enough to "
              f"send)")

        try:
            from IPython.display import Image as Shown, display as show

            show(Shown(filename=str(SHEET), width=1000))

        except Exception:
            pass

    if CHECK_DRAWINGS:

        # The drawing model has already gone; nothing is holding the
        # card while you look.
        raise SystemExit(
            f"\n\nStopped after the drawings, on purpose.\n\n"
            f"{len(SHOTS)} of them, and animating them is the next three "
            f"hours. Look at the sheet\nabove first - it is also saved "
            f"as {SHEET}.\n\n"
            "What to look for:\n\n"
            "  -  Is the whole character in frame, head to feet, in all "
            "of them?\n"
            "  -  Does each scene show what its line of the song says?\n"
            "  -  Is anyone wearing somebody else's clothes?\n"
            "  -  Any flat grey skies?\n\n"
            "Wrong ones: fix those lines in script.txt, delete just "
            "those files from\nOutput/Scenes, and run again - only the "
            "ones you deleted are drawn afresh.\n\n"
            "Happy with them: set CHECK_DRAWINGS = False at the top and "
            "run again.\nNothing is drawn twice, so it costs nothing."
        )


# ======================================================================
# The model.  Kept between runs - loading it is most of the wait
# ======================================================================

if "pipe" not in globals():

    if FAMILY == "wan":

        from diffusers import AutoencoderKLWan

        # One model, two doors. Words only, or words and a first
        # frame. Asked of the shots rather than of WRITTEN, because a
        # script with a reference picture has just been given a first
        # frame for every one of them.
        if all(shot["picture"] is None for shot in SHOTS):
            from diffusers import WanPipeline as Pipeline
        else:
            from diffusers import WanImageToVideoPipeline as Pipeline

    elif BIG:
        from diffusers import LTXConditionPipeline as Pipeline
    else:
        from diffusers import LTXImageToVideoPipeline as Pipeline

    print("\nLoading the model. A few minutes the first time"
          + (" - it is a big download." if BIG else "."))

    started = time.time()

    parts = {}

    if QUANTISE:

        from transformers import BitsAndBytesConfig, T5EncoderModel

        # 9GB of text encoder will not fit through a small card's RAM at
        # full size. In 8-bit it goes straight to the GPU at about 4.7GB.
        parts["text_encoder"] = T5EncoderModel.from_pretrained(
            MODEL,
            subfolder="text_encoder",
            quantization_config=BitsAndBytesConfig(load_in_8bit=True),
            device_map="auto",
        )

    if FAMILY == "wan":

        # The VAE is kept in float32 deliberately. Wan's own
        # instructions say to: in bfloat16 it returns blotchy colour.
        # It is small, so this costs nothing worth having.
        vae = AutoencoderKLWan.from_pretrained(
            MODEL, subfolder="vae", torch_dtype=torch.float32,
        )

        pipe = Pipeline.from_pretrained(
            MODEL, vae=vae, torch_dtype=DTYPE, **parts
        )

        try:
            from diffusers import UniPCMultistepScheduler

            pipe.scheduler = UniPCMultistepScheduler.from_config(
                pipe.scheduler.config, flow_shift=FLOW_SHIFT,
            )

        except Exception:
            # Losing the shift costs some sharpness. Stopping the run
            # over it costs an hour, so it is not worth stopping for.
            print("    (flow_shift could not be set - carrying on)")

        # Nothing here is bigger than the card on its own, so moving
        # whole components on and off is enough. None of the fp8 and
        # leaf-level work below applies.
        pipe.enable_model_cpu_offload()

    elif BIG:

        from diffusers import AutoModel
        from diffusers.hooks import apply_group_offloading

        # 13B in bfloat16 is 26GB against a 24GB card, and moving whole
        # components on and off does not help when one component is the
        # thing that does not fit. Two steps make it fit:
        #
        #   fp8 storage      - the weights sit in half the space and are
        #                      converted back a layer at a time as they
        #                      are used. Needs an Ada card or newer.
        #   leaf offloading  - only the small piece being computed is on
        #                      the card at all; the rest waits in RAM.
        transformer = AutoModel.from_pretrained(
            MODEL, subfolder="transformer", torch_dtype=DTYPE,
        )

        transformer.enable_layerwise_casting(
            storage_dtype=torch.float8_e4m3fn, compute_dtype=DTYPE,
        )

        pipe = Pipeline.from_pretrained(
            MODEL, transformer=transformer, torch_dtype=DTYPE, **parts
        )

        onload = torch.device("cuda")
        offload = torch.device("cpu")

        pipe.transformer.enable_group_offload(
            onload_device=onload,
            offload_device=offload,
            offload_type="leaf_level",
            use_stream=True,
        )

        apply_group_offloading(
            pipe.text_encoder,
            onload_device=onload,
            offload_type="block_level",
            num_blocks_per_group=2,
        )

        apply_group_offloading(
            pipe.vae, onload_device=onload, offload_type="leaf_level",
        )

    else:

        pipe = Pipeline.from_pretrained(MODEL, torch_dtype=DTYPE, **parts)

        if QUANTISE:
            # The text encoder is already on the card; move what is left.
            pipe.transformer.to("cuda")
            pipe.vae.to("cuda")
        else:
            pipe.to("cuda")

    # The 13B has placed itself: nothing sits on the card until it is
    # needed, so there is nothing to move here.

    # Decoding every frame in one piece is what runs a card out of
    # memory. Tiling decodes it in patches instead.
    pipe.vae.enable_tiling()

    print(f"Model ready in {time.time() - started:.0f}s "
          f"({torch.cuda.memory_allocated() / 1e9:.1f}GB on the card).")

else:
    print("\nModel already loaded - reusing it.")


# ======================================================================
# One picture -> one clip
# ======================================================================

def fitted(picture):
    """Cover the frame and crop the overflow, rather than squash."""

    scale = max(WIDTH / picture.width, HEIGHT / picture.height)

    picture = picture.resize(
        (round(picture.width * scale), round(picture.height * scale)),
        Image.LANCZOS,
    )

    left = (picture.width - WIDTH) // 2
    top = (picture.height - HEIGHT) // 2

    return picture.crop((left, top, left + WIDTH, top + HEIGHT))


def animate(shot, clip_file, seconds):
    """
    One shot -> one clip of exactly `seconds`.

    A shot either starts from a picture or from a line of your script.
    Everything after that is the same either way.

    The model is always asked for the same short clip, however long the
    shot has to be on screen. Asking it for a long one is what made the
    face melt. The clip is then played forwards, backwards and forwards
    again until the time is filled: a sway reads as continuous that way,
    and the seam is at the moment the movement turns around, where it is
    least visible.

    Returns (frames, loops) - what was generated, and how many times the
    forward-and-back pair had to run.
    """

    image = (
        fitted(Image.open(shot["picture"]).convert("RGB"))
        if shot["picture"] is not None else None
    )

    prompt = shot["prompt"]

    def run(count):

        asked = dict(
            prompt=prompt,
            negative_prompt=NEGATIVE,
            width=WIDTH,
            height=HEIGHT,
            num_frames=count,
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE,
            generator=torch.Generator("cpu").manual_seed(42),
        )

        # Written scenes send no picture at all: the pipeline makes the
        # whole thing from the words. Same model, same everything else.
        if image is not None:
            asked["image"] = image

        if FAMILY == "wan":

            # Wan has no frame_rate argument - 24fps is what the 5B was
            # trained at and what it always makes - and it reads a
            # longer prompt than LTX does.
            asked["max_sequence_length"] = 512

        else:

            asked["frame_rate"] = FPS

            # Stated rather than left to a default. The tokenizer warns
            # about 128 because that is its own configured length; the
            # pipeline allows 256, and a full prompt here is about 180.
            # Left implicit, a future default could silently cut the
            # style words off the end - where they sit.
            asked["max_sequence_length"] = 256

            if BIG:
                asked.update(
                    timesteps=TIMESTEPS,
                    decode_timestep=0.05,
                    decode_noise_scale=0.025,
                )

                if image is not None:
                    # No noise on the conditioning picture. This is the
                    # setting that keeps the clip on the picture it was
                    # given, and the 2B pipeline has no equivalent.
                    asked["image_cond_noise_scale"] = 0.0

        return pipe(**asked).frames[0]

    frames = MAX_FRAMES

    try:
        video = run(frames)

    except torch.cuda.OutOfMemoryError:

        gc.collect()
        torch.cuda.empty_cache()

        frames = max(25, ((frames // 2 - 1) // 8) * 8 + 1)

        print(f"    card ran out of room - shorter clip ({frames} frames)")

        try:
            video = run(frames)

        except torch.cuda.OutOfMemoryError as full:

            # A shorter clip does not help when it is the model that
            # will not fit. Say what to do rather than repeat 34 frames
            # of traceback.
            raise SystemExit(
                "The card is out of memory even on the shortest clip.\n\n"
                + (
                    "Either lower CLIP_SECONDS and the size at the top,"
                    '\nor set FORCE_MODEL = "small" for the 2B, which '
                    "fits anywhere.\n\n"
                    if BIG else
                    "Lower CLIP_SECONDS at the top of this cell, or use "
                    "a bigger GPU.\n\n"
                )
                + "Runtime > Restart session first - a half loaded model "
                "is still holding the card.\n\n"
                "Clips already finished are safe in Output/Clips and will "
                "not be made again."
            ) from full

    from diffusers.utils import export_to_video

    from diffusers.utils import export_to_video

    raw = clip_file.with_name(clip_file.stem + "_raw.png.mp4")

    export_to_video(video, str(raw), fps=FPS)

    # ------------------------------------------------- the model's mark

    # LTX 0.9.8 distilled was trained on captioned video and stamps a
    # line of garbled text along the bottom of everything it makes. No
    # negative prompt shifts it. Cutting the band off is what works, and
    # the edit re-frames afterwards so nothing looks short of picture.
    keep = HEIGHT - (round(HEIGHT * CROP_BOTTOM / 100) // 2) * 2

    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(raw),
            "-vf", f"crop={WIDTH}:{keep}:0:0",
            "-r", str(FPS),
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-preset", "veryfast", "-crf", "16",
            str(clip_file),
        ],
        check=True,
    )

    raw.unlink(missing_ok=True)

    return frames


# ======================================================================
# Every picture
# ======================================================================

# A clip is reused only when the thing it was made from has not moved.
# Skipping by file name alone is what would quietly leave you with a clip
# of last week's Scene02 after you replaced the picture.
def side_by_side(sources, target):
    """
    Two or three clips in one frame, cut down the middle.

    Each is cropped to its share of the width from the centre, which is
    where the character is - every drawing is framed with one subject
    in the middle, so the middle is what survives a crop.
    """

    across = len(sources)

    # Even numbers, and the last panel takes the remainder so the three
    # of them add up to exactly the frame and not a pixel less.
    panel = (WIDTH // across) // 2 * 2

    widths = [panel] * (across - 1) + [WIDTH - panel * (across - 1)]

    chain = []

    for number, wide in enumerate(widths):
        chain.append(
            f"[{number}:v]crop=iw/{across}:ih:(iw-iw/{across})/2:0,"
            f"scale={wide}:{HEIGHT}:flags=lanczos[p{number}]"
        )

    chain.append(
        "".join(f"[p{n}]" for n in range(across))
        + f"hstack=inputs={across}[out]"
    )

    command = ["ffmpeg", "-y", "-loglevel", "error"]

    for source in sources:
        command += ["-i", str(source)]

    command += [
        "-filter_complex", ";".join(chain),
        "-map", "[out]",
        "-r", str(FPS),
        "-c:v", "libx264", "-pix_fmt", "yuv420p",
        "-preset", "veryfast", "-crf", "16",
        str(target),
    ]

    subprocess.run(command, check=True)

    return target


STAMPS = CLIPS / "made.json"

try:
    stamps = json.loads(STAMPS.read_text(encoding="utf-8"))
except Exception:
    stamps = {}


def stamp_for(shot):

    # What the clip was generated from - and nothing about the song or
    # the edit. Swapping the song used to count as a change and threw
    # away every clip, when the clips would have been identical.
    made_from = {
        "prompt": shot["prompt"],
        "size": f"{WIDTH}x{HEIGHT}",
        "frames": MAX_FRAMES,
        "model": MODEL,
    }

    if shot["picture"] is not None:

        facts = shot["picture"].stat()

        made_from.update(
            picture=shot["picture"].name,
            bytes=facts.st_size,
            modified=int(facts.st_mtime),
        )

    return made_from


clip_of = {}

# The generated ones first, all of them, and the side-by-side ones
# after - so a "Split of" line may name any scene in the script,
# including one that comes later in the song.
ANIMATED = [shot for shot in SHOTS if not shot["split"]]

for number, shot in enumerate(ANIMATED, start=1):

    clip_file = CLIPS / f"{shot['name']}.mp4"

    label = f"[{number}/{len(ANIMATED)}] {shot['name']}"

    wanted = stamp_for(shot)

    finished = clip_file.exists() and clip_file.stat().st_size > 0

    if finished and stamps.get(clip_file.name) == wanted:
        print(f"{label}: already made, skipping")
        clip_of[shot["name"]] = clip_file
        continue

    if finished:
        print(f"{label}: changed since last time - making it again")

    print(f"{label}: animating ...")

    started = time.time()

    frames = animate(shot, clip_file, SHARE)

    took = (time.time() - started) / 60

    print(f"    done in {took:.1f} min ({frames / FPS:.1f}s of movement)")

    # Measured, not guessed, and said once - after the first clip, when
    # there is still time to stop and turn DRAFT on instead of finding
    # out an hour later how long an hour is.
    left = len(ANIMATED) - number

    if number == 1 and left:
        print(f"    {left} to go, so about {took * left:.0f} more "
              f"minutes at this rate."
              + ("" if QUALITY == "draft" else
                 ' QUALITY = "draft" is roughly a third of that.'))

    # Written after every clip, not at the end: a session that dies has
    # to leave behind an honest record of what is really finished.
    stamps[clip_file.name] = wanted

    STAMPS.write_text(json.dumps(stamps, indent=1), encoding="utf-8")

    clip_of[shot["name"]] = clip_file


# ---------------------------------------------------- side by side

for shot in SHOTS:

    if not shot["split"]:
        continue

    clip_file = CLIPS / f"{shot['name']}.mp4"

    sources = [clip_of[name] for name in shot["split"]]

    wanted = {
        "split": shot["split"],
        "size": f"{WIDTH}x{HEIGHT}",
        "from": [int(source.stat().st_mtime) for source in sources],
    }

    if (clip_file.exists() and clip_file.stat().st_size > 0
            and stamps.get(clip_file.name) == wanted):
        clip_of[shot["name"]] = clip_file
        continue

    print(f"{shot['name']}: {' + '.join(shot['split'])} side by side")

    side_by_side(sources, clip_file)

    stamps[clip_file.name] = wanted

    STAMPS.write_text(json.dumps(stamps, indent=1), encoding="utf-8")

    clip_of[shot["name"]] = clip_file


# Back into the order the script is written in, whichever way each one
# was made.
made = [clip_of[shot["name"]] for shot in SHOTS]


# ======================================================================
# The edit
# ======================================================================
#
# This is the part that decides whether the video looks made or
# assembled, and it costs no GPU at all.
#
# A children's channel does not put one picture on screen for eleven
# seconds. It cuts every two to four seconds, it lands every cut on the
# beat of the song, and it comes back to the same setup later from a
# different angle. Eleven clips become forty shots that way, and the
# song carries the rhythm of the edit.
#
# It is also, by luck, exactly what this model needs: nothing is on
# screen long enough to drift.

def beats_of(song):
    """
    Where the beats fall, in seconds.

    Falls back to an even grid when the song cannot be analysed - a
    steady edit is still better than one long held shot.
    """

    if song is None:
        return []

    try:
        import librosa

        samples, rate = librosa.load(str(song), sr=22050, mono=True)

        _, frames = librosa.beat.beat_track(y=samples, sr=rate)

        times = librosa.frames_to_time(frames, sr=rate)

        return [float(t) for t in times]

    except Exception as trouble:

        print(f"  (could not find the beat - {type(trouble).__name__} - "
              "cutting on an even grid instead)")

        return []


def spread_over(lines, beats, seconds):
    """
    When each line of the song is on screen.

    Split on the beat, the same way the cuts are, so the words change
    with the music instead of on a stopwatch. Without beats it is an
    even share.
    """

    if not lines:
        return []

    usable = [t for t in beats if 0 < t < seconds]

    if len(usable) < len(lines):
        step = seconds / len(lines)
        edges = [step * n for n in range(len(lines) + 1)]

    else:
        # The beat nearest each line's fair share of the song.
        edges = [0.0]
        for n in range(1, len(lines)):
            ideal = seconds * n / len(lines)
            edges.append(min(usable, key=lambda t: abs(t - ideal)))
        edges.append(seconds)

    return list(zip(lines, edges[:-1], edges[1:]))


def cuts_on_words(timed, seconds):
    """
    Where the shots change when there are words to change them with.

    A cut on a beat is right, but the beat is not the thing anyone
    hears change - the line of the song is. Cutting where a line begins
    means the picture changes exactly when the words do, which is the
    whole of what "the lyrics match the movement" asks for.

    Lines are gathered up if they are too quick to hold a shot, and a
    line that outstays a shot is divided inside itself.
    """

    edges = [0.0]

    for _, begins, _ends in timed:
        if begins - edges[-1] >= SHOT_SECONDS * 0.6:
            edges.append(begins)

    if seconds - edges[-1] < SHOT_SECONDS / 2 and len(edges) > 1:
        edges.pop()

    edges.append(seconds)

    points = [edges[0]]

    for point in edges[1:]:

        begins = points[-1]
        span = point - begins

        if span > SHOT_SECONDS * 1.8:

            inside = max(2, round(span / SHOT_SECONDS))

            for n in range(1, inside):
                points.append(begins + span * n / inside)

        points.append(point)

    return points


def cuts_for(beats, seconds):
    """
    Where the shots change, in seconds, from 0 to the end.

    Beats are gathered up until a shot has run about SHOT_SECONDS, so
    every cut lands on one. Without beats it is an even grid.

    Returns (points, on_beat), because saying "cut on the beat" when it
    was really a stopwatch is the sort of thing you then believe.
    """

    beats = [t for t in beats if 0 < t < seconds]

    if not beats:
        count = max(1, round(seconds / SHOT_SECONDS))
        step = seconds / count
        return [step * n for n in range(count + 1)], False

    points = [0.0]

    for beat in beats:
        if beat - points[-1] >= SHOT_SECONDS:
            points.append(beat)

    # A last shot shorter than half a shot is a flash; give its time to
    # the one before instead.
    if seconds - points[-1] < SHOT_SECONDS / 2 and len(points) > 1:
        points.pop()

    points.append(seconds)

    return points, True


# ------------------------------------------------------------- camera

# Every shot moves. A still frame from a video model reads as a freeze,
# and a slow push or drift is what the eye expects from animation. They
# cycle, so returning to the same clip does not look like a repeat.
MOVES = ("push_in", "pan_right", "hold", "pull_out", "pan_left", "push_in_up")


def pulse_at(beats, frames, strength=0.05, fall=0.18):
    """
    A little push on every beat, as a zoompan expression.

    This is what "movement with the song" actually means. The cuts
    already land on the beat, but between cuts nothing was answering
    the music - and this model gives so little movement of its own
    that the picture sat there.

    Each beat adds a small bump that falls away over `fall` seconds, so
    the frame breathes in time rather than drifting past the rhythm.

    Measured against a still picture: the camera move alone comes out
    at 1.81, and with this it is 4.26 - more than double, with the
    peaks landing on the beats. 0.08 was jumpy.
    """

    if not beats:
        return ""

    terms = []

    for beat in beats:
        # `on` is the output frame, so on/FPS is the second we are at.
        terms.append(
            f"{strength}*max(0\\,1-abs(on/{FPS}-{beat:.3f})/{fall})"
        )

    return "+" + "+".join(terms)


def move_filter(move, frames, beats=()):
    """
    A slow camera move across the clip, ending up at the output size.

    The clip is scaled well past the output first: zoompan works in
    whole source pixels, so moving across a frame-sized picture makes
    the movement visibly step.
    """

    # Only a little past the output. The clip itself is 960 wide, so
    # scaling to 4K before moving across it buys nothing that is really
    # there and costs minutes per shot - and on Colab minutes are money.
    big_w = round(OUTPUT_WIDTH * 1.2 / 2) * 2
    big_h = round(OUTPUT_HEIGHT * 1.2 / 2) * 2

    last = max(1, frames - 1)

    # 1.16 was a lurch and 1.06 was nothing: with it the whole video
    # measured a third of the movement it had before, because the model
    # itself barely moves and the camera was carrying all of it. This
    # is the middle, and the beat pulse below is what actually makes it
    # feel animated.
    zoom = 1.0 if move == "hold" else 1.11

    centre_x = "iw/2-(iw/zoom/2)"
    centre_y = "ih/2-(ih/zoom/2)"
    span_x = "(iw-iw/zoom)"
    span_y = "(ih-ih/zoom)"

    growing = f"1+{zoom - 1:.6f}*on/{last}"
    shrinking = f"{zoom:.6f}-{zoom - 1:.6f}*on/{last}"
    held = f"{zoom:.6f}"

    moves = {
        # A held shot is still scaled and cropped, just not moved.
        "hold":       (f"{1.0:.6f}", centre_x, centre_y),
        "push_in":    (growing, centre_x, centre_y),
        "pull_out":   (shrinking, centre_x, centre_y),
        "pan_right":  (held, f"{span_x}*on/{last}", centre_y),
        "pan_left":   (held, f"{span_x}*(1-on/{last})", centre_y),
        "push_in_up": (growing, centre_x, f"{span_y}*(1-on/{last})"),
    }

    expression, x, y = moves.get(move, moves["push_in"])

    # The beat rides on top of whatever the camera was already doing.
    expression = f"({expression}){pulse_at(beats, frames)}"

    return (
        # lanczos. This used to be bicubic, and the reason given was
        # that a 960 wide clip has no detail for a sharper filter to
        # find. The clip is 1024 or 1280 wide now and the finished
        # video is 1920, so there is, and this is the last chance to
        # keep it.
        f"scale={big_w}:{big_h}:force_original_aspect_ratio=increase:"
        f"flags=lanczos,"
        f"crop={big_w}:{big_h},"
        f"zoompan=z='{expression}':x='{x}':y='{y}'"
        f":d=1:s={OUTPUT_WIDTH}x{OUTPUT_HEIGHT}:fps={FPS},"
        f"format=yuv420p"
    )


def bounce_of(clip_file):
    """
    The clip forwards then backwards, so it can run for ever.

    A frame is dropped at each end of the reversed half: the first is
    the one the forward half just finished on and the last is the one
    the loop is about to start on again, and holding either for two
    frames shows as a hitch.
    """

    # Kept in the Edit folder, which is emptied every run. Keeping it
    # beside the clip meant a regenerated clip quietly kept the loop
    # made from the old one - and the edit would still be showing last
    # week's picture.
    bounced = EDIT / f"{clip_file.stem}_loop.mp4"

    frames = MAX_FRAMES

    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(clip_file),
            "-filter_complex",
            "[0:v]split[fwd][back];"
            f"[back]reverse,trim=start_frame=1:end_frame={frames - 1},"
            "setpts=PTS-STARTPTS[rev];"
            "[fwd][rev]concat=n=2:v=1[out]",
            "-map", "[out]",
            "-r", str(FPS),
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-preset", "veryfast", "-crf", "16",
            str(bounced),
        ],
        check=True,
    )

    return bounced


def cut_shot(source, target, seconds, move, beats=()):
    """One shot of the finished video: looped to length, and moving."""

    frames = max(2, round(seconds * FPS))

    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-stream_loop", "-1", "-i", str(source),
            "-t", f"{seconds:.3f}",
            "-vf", move_filter(move, frames, beats),
            "-r", str(FPS),
            # A working file, encoded fast and near-losslessly. The one
            # encode that matters is the last one.
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-preset", "ultrafast", "-crf", "14",
            str(target),
        ],
        check=True,
    )


# ======================================================================
# Cut it together
# ======================================================================

EDIT = OUTPUT / "Edit"

if EDIT.exists():
    shutil.rmtree(EDIT)

EDIT.mkdir(parents=True)

length = SONG_SECONDS or (SHARE * len(made))

BEATS = beats_of(SONG)

points, on_beat = cuts_for(BEATS, length)

# The words win where there are words. Their own timings come from the
# beats already, so this is still cut on the beat - it is cut on the
# beats that a line of the song actually starts on.
WORD_CUT = False

SPANS = []

if LYRICS:

    SPANS = spread_over(LYRICS, BEATS, length)

    on_words = cuts_on_words(SPANS, length)

    if len(on_words) > 2:
        points, WORD_CUT = on_words, True

print(f"\nEditing: {len(points) - 1} shot(s) across {length:.1f}s"
      + (", cut where each line of the song starts" if WORD_CUT else
         ", cut on the beat" if on_beat else ", cut on an even grid"))

longest_cut = max(points[n + 1] - points[n]
                  for n in range(len(points) - 1))

# A clip long enough to cover the longest cut on its own is left
# alone. Bouncing it would turn the movement round in the middle of a
# shot, which is a large part of why these videos read as a wobble
# rather than a walk.
if MAX_FRAMES / FPS >= longest_cut:
    print("         clips run forwards only - no reversing")
    loops = made
else:
    loops = [bounce_of(clip) for clip in made]

pieces = []

for number in range(len(points) - 1):

    start, finish = points[number], points[number + 1]

    # Which shot this cut belongs to.
    #
    # One scene per line of the song is the case worth having, because
    # then scene four IS line four and the picture shows what is being
    # sung. Without it the shots are shared out by time, and the first
    # full video showed what that costs: "Meow, meow, meow!" over a
    # duckling and "Quack, quack, quack!" over a puppy, all the way
    # through. Nothing in the words was wrong and nothing in the
    # pictures was wrong - they were simply not the same story.
    if LINE_FOR_LINE:

        which = min(
            len(loops) - 1,
            max((n for n, (_line, began, _ends) in enumerate(SPANS)
                 if began <= start), default=0),
        )

    else:
        # Each shot gets its own stretch of the song and is cut up
        # several times inside it - not the cut number cycled round the
        # shots, which would throw away the order your script is in.
        which = min(len(loops) - 1, int(start / length * len(loops)))

    source = loops[which]

    move = MOVES[number % len(MOVES)]

    piece = EDIT / f"{number:03d}.mp4"

    # The beats inside this shot, counted from the shot's own start,
    # because the filter sees a clip that begins at zero.
    inside = [b - start for b in BEATS if start < b < finish]

    cut_shot(source, piece, finish - start, move, inside)

    pieces.append(piece)

print(f"         {len(pieces)} cuts from {len(loops)} clip(s), "
      f"in the order you wrote them")


# ======================================================================
# The words of the song, on screen
# ======================================================================
#
# This is the one thing that makes a rhyme video feel like it is *of*
# the song rather than merely over it - children sing along with it, and
# it is what every channel in this corner of YouTube does.
#
# It costs no GPU at all. Put a lyrics.txt in the Input folder, one line
# a line, and the lines are spread across the song on its beats.

def font_for(text):
    """
    A font that can actually draw these words.

    Devanagari in a font that has no Devanagari is a row of empty
    boxes, and it is better to say so than to render that.
    """

    wanted = ["Noto Sans Devanagari", "Lohit Devanagari", "Mukta",
              "Noto Sans", "DejaVu Sans", "Liberation Sans"]

    try:
        listed = subprocess.run(
            ["fc-list", "--format", "%{family}\n"],
            capture_output=True, text=True, timeout=30,
        ).stdout

    except Exception:
        listed = ""

    families = {name.strip() for line in listed.splitlines()
                for name in line.split(",")}

    devanagari = any("\u0900" <= ch <= "\u097f" for ch in text)

    for name in wanted:

        if name not in families:
            continue

        if devanagari and "Devanagari" not in name and "Noto Sans" != name:
            continue

        return name, True

    if devanagari:
        print("  ! The lyrics are in Devanagari and no font here can "
              "draw it - they would\n    come out as empty boxes, so "
              "they are being left off. In cell 1 add:\n"
              "        !apt-get -qq install -y fonts-indic")
        return None, False

    return "DejaVu Sans", True


def timecode(seconds):
    """0:00:01.50, which is what ASS wants."""

    hours, rest = divmod(max(0.0, seconds), 3600)
    minutes, rest = divmod(rest, 60)

    return f"{int(hours)}:{int(minutes):02d}:{rest:05.2f}"


def write_lyrics(timed, target, width, height, font):
    """An ASS subtitle file, styled for a children's video."""

    # Bigger and blacker than it was. The first finished frame had
    # "Wag your tail," on it in white with a thin grey edge, over lit
    # grass, and you could barely see it - the contrast had to be
    # pushed 2x in an editor before it read at all.
    #
    # What the children's channels do is not subtle and should not be:
    # heavy, white, and a black outline thick enough that the words
    # keep their shape over a bright sky and over dark grass alike.
    size = round(height * 0.085)
    margin = round(height * 0.075)

    escaped = str(target)

    target.write_text(
        "[Script Info]\n"
        "ScriptType: v4.00+\n"
        f"PlayResX: {width}\n"
        f"PlayResY: {height}\n"
        "WrapStyle: 0\n"
        "\n"
        "[V4+ Styles]\n"
        "Format: Name, Fontname, Fontsize, PrimaryColour, OutlineColour, "
        "BackColour, Bold, BorderStyle, Outline, Shadow, Alignment, "
        "MarginL, MarginR, MarginV, Encoding\n"
        # Pure black outline, not the dark grey this used to be, and
        # twice as thick. A drop shadow under it as well, so the words
        # sit above the picture rather than in it.
        f"Style: Sing,{font},{size},&H00FFFFFF,&H00000000,&H90000000,"
        f"-1,1,{max(4, round(size * 0.14))},4,2,"
        f"{round(width * 0.06)},{round(width * 0.06)},{margin},1\n"
        "\n"
        "[Events]\n"
        "Format: Layer, Start, End, Style, Name, MarginL, MarginR, "
        "MarginV, Effect, Text\n"
        + "".join(
            f"Dialogue: 0,{timecode(start)},{timecode(end)},Sing,,0,0,0,,"
            "{\\fad(120,120)}" + line.replace("\n", " ") + "\n"
            for line, start, end in timed
        ),
        encoding="utf-8",
    )

    return target


# ======================================================================
# The finished file
# ======================================================================

listing = OUTPUT / "shots.txt"

listing.write_text(
    "".join(f"file '{piece}'\n" for piece in pieces),
    encoding="utf-8",
)

FINAL = OUTPUT / "Episode.mp4"

# The wide video and the vertical one each need the words drawn at their
# own size, so the master is made without them and each burns its own.
MASTER = OUTPUT / "_master.mp4" if LYRICS else FINAL

command = [
    "ffmpeg", "-y", "-loglevel", "error",
    "-f", "concat", "-safe", "0", "-i", str(listing),
]

if SONG:
    command += [
        "-i", str(SONG),
        # YouTube turns everything down to about -14 LUFS. A quiet
        # upload stays quiet next to a channel that mastered theirs.
        "-af", "loudnorm=I=-14:TP=-1.5:LRA=11",
        "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2",
        "-shortest",
    ]

command += [
    "-c:v", "libx264", "-profile:v", "high", "-level", "4.0",
    "-pix_fmt", "yuv420p",
    "-preset", "medium", "-crf", "18",
    "-r", str(FPS),
    "-movflags", "+faststart",
    str(MASTER),
]

subprocess.run(command, check=True)

listing.unlink(missing_ok=True)


# The vertical cut.
#
# This was a centre crop, then a fit-with-blurred-background, and the
# fit was worse: a 16:9 frame inside 9:16 fills under a third of the
# height, so most of a Short was a huge blurred face with a thin strip
# of video in the middle.
#
# So: crop, but from a frame whose camera is now calm. The character is
# in the middle of every shot because that is how the clips are framed,
# so the middle is what to keep.
# No output label: the caller adds one, because the lyrics have to be
# chained on after the crop.
VERTICAL = (
    "[0:v]crop=trunc(ih*9/16/2)*2:ih,"
    "scale=1080:1920:flags=lanczos"
)


def lyric_filter(width, height, seconds):
    """
    The filter that draws the words, or "" if they cannot be drawn.

    The path goes inside a filter argument, where a colon separates
    options - so it has to be escaped or a Windows drive letter reads
    as two options.
    """

    font, usable = font_for(" ".join(LYRICS))

    if not usable:
        return ""

    sheet = write_lyrics(
        spread_over(LYRICS, BEATS, seconds),
        OUTPUT / f"lyrics_{width}x{height}.ass",
        width, height, font,
    )

    escaped = str(sheet).replace("\\", "/").replace(":", "\\:")

    return f"subtitles='{escaped}'"


def encode(source, target, chain, seconds=None, complex_chain=False):
    """One re-encode with a filter chain, kept in one place."""

    command = ["ffmpeg", "-y", "-loglevel", "error", "-i", str(source)]

    if seconds:
        command += ["-t", str(seconds)]

    if chain:
        command += (["-filter_complex", chain, "-map", "[out]", "-map", "0:a?"]
                    if complex_chain else ["-vf", chain])

    command += [
        "-c:v", "libx264", "-profile:v", "high", "-pix_fmt", "yuv420p",
        "-preset", "medium", "-crf", "18",
        "-c:a", "aac", "-b:a", "192k",
        "-movflags", "+faststart",
        str(target),
    ]

    subprocess.run(command, check=True)


if LYRICS:

    words = lyric_filter(OUTPUT_WIDTH, OUTPUT_HEIGHT, seconds_of(MASTER))

    if words:
        encode(MASTER, FINAL, words)
        print("\nLyrics  : drawn on, changing on the beat")
    else:
        shutil.copyfile(MASTER, FINAL)

print(f"\nFinished: {FINAL}")
print(f"Length  : {seconds_of(FINAL):.1f}s at {OUTPUT_WIDTH}x{OUTPUT_HEIGHT}")

# "It is still not right" is not something anyone can act on, and a
# video is not always something you can send. These two numbers are.
#
# They separate the two possible faults, which need opposite fixes:
# the model not moving anything, and the edit not doing enough with
# what it was given.
FROM_MODEL = motion_of(made[0]) if made else 0.0

FINISHED = motion_of(FINAL)

print(f"Movement: {FINISHED:.1f} in the finished video, "
      f"{FROM_MODEL:.1f} from the model alone")
print( "          under 5 is a slideshow, 10 is alive, 15+ is what the "
       "big children's")
print( "          channels run at. Send both numbers - they say more "
       "than a description.")

# ------------------------------------------------------------- Shorts

if MAKE_SHORT and not TEST_ONE_PICTURE:

    SHORT = OUTPUT / "Episode_Short.mp4"

    # Built from the master, not from the finished wide video: the words
    # have to be drawn at the vertical size, and burning them twice
    # would show the wide ones shrunk underneath.
    vertical = VERTICAL

    if LYRICS:
        tall_words = lyric_filter(1080, 1920, min(55.0, seconds_of(MASTER)))

        if tall_words:
            vertical = f"{VERTICAL}[fitted];[fitted]{tall_words}[out]"
        else:
            vertical = f"{VERTICAL}[out]"
    else:
        vertical = f"{VERTICAL}[out]"

    encode(MASTER, SHORT, vertical, seconds=55, complex_chain=True)

    print(f"Shorts  : {SHORT} ({seconds_of(SHORT):.0f}s, 1080x1920)")

# ------------------------------------------------------------ Preview

# A copy small enough to send.
#
# The finished file is 1080p and a couple of hundred megabytes, and
# most places will not take one that size - which meant the only thing
# that ever came back was a description, and a description cannot be
# measured. This one is made small enough to go anywhere, and made
# smaller again if the first try is not.
PREVIEW = OUTPUT / "Episode_Preview.mp4"

for PREVIEW_WIDTH, quality in ((640, 30), (480, 34), (384, 38)):

    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error", "-i", str(FINAL),
            "-vf", f"scale={PREVIEW_WIDTH}:-2",
            "-c:v", "libx264", "-preset", "veryfast", "-crf", str(quality),
            "-c:a", "aac", "-b:a", "64k", "-ac", "1",
            "-movflags", "+faststart",
            str(PREVIEW),
        ],
        check=True,
    )

    if PREVIEW.stat().st_size < 25e6:
        break

print(f"Preview : {PREVIEW} "
      f"({PREVIEW.stat().st_size / 1e6:.0f}MB, {PREVIEW_WIDTH} wide "
      "- small enough to send anywhere)")

if not ON_DRIVE and colab_files is not None:

    # Nothing here survives the session, so hand the file over rather
    # than leave it somewhere that is about to be deleted.
    print("\nDownloading it to your PC - the session keeps no copy.")

    colab_files.download(str(FINAL))

    colab_files.download(str(PREVIEW))

    if MAKE_SHORT and not TEST_ONE_PICTURE:
        colab_files.download(str(SHORT))

if LYRICS and MASTER != FINAL:
    MASTER.unlink(missing_ok=True)

from IPython.display import Video, display

display(Video(str(FINAL), embed=True, width=720))

# ----------------------------------------------------------------------
# The file is in Drive, so it is already on your PC if Drive syncs there.
#
# Want one shot to do something else? Change its line in script.txt, or
# add the picture to PROMPTS at the top, and run this cell again. Only
# what changed is generated afresh - the rest are kept, and the edit is
# rebuilt every time because cutting costs nothing.
# ----------------------------------------------------------------------
